# Angio-VLM: прогон на Colab (всё в одном)

1. **Среда выполнения → Сменить среду выполнения → T4 GPU** (не TPU).
2. **Среда выполнения → Выполнить всё.**
3. В конце скачается архив `results.zip`, его нужно прислать.

Весь код встроен в ноутбук, репозиторий не нужен.

In [ ]:
!nvidia-smi -L
!pip install -q -U transformers accelerate bitsandbytes pycocotools rank_bm25

In [ ]:
!mkdir -p /content/angio-vlm/src /content/angio-vlm/kb /content/angio-vlm/results

In [ ]:
%%writefile /content/angio-vlm/src/gt.py
"""Build structured ground truth from ARCADE COCO annotations.

ARCADE has two independent subsets:
  syntax/   - 25 coronary segments (SYNTAX numbering), no stenosis labels
  stenosis/ - only stenosis polygons, no segment labels
So artery-level facts are evaluated on syntax/, stenosis facts on stenosis/.
"""
import json
from pathlib import Path

import numpy as np
from pycocotools import mask as mask_utils

# SYNTAX segment -> artery
SEGMENT_TO_ARTERY = {
    "1": "RCA", "2": "RCA", "3": "RCA", "4": "RCA",
    "16": "RCA", "16a": "RCA", "16b": "RCA", "16c": "RCA",
    "5": "LM",
    "6": "LAD", "7": "LAD", "8": "LAD", "9": "LAD", "9a": "LAD", "10": "LAD", "10a": "LAD",
    "11": "LCx", "12": "LCx", "12a": "LCx", "12b": "LCx", "13": "LCx",
    "14": "LCx", "14a": "LCx", "14b": "LCx", "15": "LCx",
}
ARTERIES = ["RCA", "LM", "LAD", "LCx"]


def load_coco(root, task, split):
    path = Path(root) / task / split / "annotations" / f"{split}.json"
    d = json.load(open(path))
    cats = {c["id"]: c["name"] for c in d["categories"]}
    imgs = {im["id"]: im for im in d["images"]}
    anns = {}
    for a in d["annotations"]:
        anns.setdefault(a["image_id"], []).append(a)
    return imgs, anns, cats


def ann_mask(a, h, w):
    rles = mask_utils.frPyObjects(a["segmentation"], h, w)
    return mask_utils.decode(mask_utils.merge(rles)).astype(bool)


def build_syntax_gt(root, split):
    imgs, anns, cats = load_coco(root, "syntax", split)
    out = []
    for iid, im in sorted(imgs.items()):
        h, w = im["height"], im["width"]
        segs = sorted({cats[a["category_id"]] for a in anns.get(iid, [])},
                      key=lambda s: (int("".join(c for c in s if c.isdigit())), s))
        arteries = sorted({SEGMENT_TO_ARTERY[s] for s in segs}, key=ARTERIES.index)
        side = "right" if "RCA" in arteries else "left"
        out.append({
            "image_id": iid, "file": str(Path(root) / "syntax" / split / "images" / im["file_name"]),
            "width": w, "height": h, "side": side, "arteries": arteries, "segments": segs,
        })
    return out


def syntax_artery_masks(root, split, image_id):
    imgs, anns, cats = load_coco(root, "syntax", split)
    im = imgs[image_id]
    masks = {}
    for a in anns.get(image_id, []):
        art = SEGMENT_TO_ARTERY[cats[a["category_id"]]]
        m = ann_mask(a, im["height"], im["width"])
        masks[art] = masks.get(art, np.zeros_like(m)) | m
    return masks


def build_stenosis_gt(root, split):
    imgs, anns, cats = load_coco(root, "stenosis", split)
    out = []
    for iid, im in sorted(imgs.items()):
        boxes = [a["bbox"] for a in anns.get(iid, [])]  # x, y, w, h
        out.append({
            "image_id": iid, "file": str(Path(root) / "stenosis" / split / "images" / im["file_name"]),
            "width": im["width"], "height": im["height"], "stenosis_boxes": boxes,
        })
    return out


if __name__ == "__main__":
    import argparse, collections
    ap = argparse.ArgumentParser()
    ap.add_argument("--root", default="/home/user/data/arcade")
    ap.add_argument("--out", default="/home/user/angio-vlm/results/gt_stats.json")
    args = ap.parse_args()
    stats = {}
    for split in ["train", "val", "test"]:
        sg = build_syntax_gt(args.root, split)
        st = build_stenosis_gt(args.root, split)
        seg_c = collections.Counter(s for g in sg for s in g["segments"])
        stats[split] = {
            "syntax_images": len(sg),
            "side": dict(collections.Counter(g["side"] for g in sg)),
            "artery_freq": dict(collections.Counter(a for g in sg for a in g["arteries"])),
            "segments_per_image_mean": round(float(np.mean([len(g["segments"]) for g in sg])), 2),
            "segment_freq": dict(seg_c.most_common()),
            "stenosis_images": len(st),
            "stenoses_per_image_mean": round(float(np.mean([len(g["stenosis_boxes"]) for g in st])), 2),
            "stenosis_box_area_frac_mean": round(float(np.mean(
                [sum(b[2] * b[3] for b in g["stenosis_boxes"]) / (g["width"] * g["height"]) for g in st])), 4),
        }
    json.dump(stats, open(args.out, "w"), indent=1, ensure_ascii=False)
    print(json.dumps(stats["test"], indent=1))


In [ ]:
%%writefile /content/angio-vlm/src/rag.py
"""Minimal BM25 retriever over the angiography knowledge base (kb/knowledge.jsonl)."""
import json
import re

from rank_bm25 import BM25Okapi


def tokenize(s):
    return re.findall(r"[a-z0-9]+", s.lower())


class Retriever:
    def __init__(self, path):
        self.chunks = [json.loads(l) for l in open(path) if l.strip()]
        self.bm25 = BM25Okapi([tokenize(c["text"] + " " + c["topic"]) for c in self.chunks])

    def search(self, query, k=6):
        scores = self.bm25.get_scores(tokenize(query))
        order = sorted(range(len(self.chunks)), key=lambda i: -scores[i])[:k]
        return [self.chunks[i] for i in order]


if __name__ == "__main__":
    r = Retriever("kb/knowledge.jsonl")
    for c in r.search("left right coronary injection laterality LM LAD LCx RCA segments course "
                      "stenosis narrowing location projection report not visible"):
        print(c["id"])


In [ ]:
%%writefile /content/angio-vlm/src/run_vlm.py
"""Zero-shot structured description of ARCADE angiograms with an open VLM, with or without RAG.

Usage:
  python src/run_vlm.py --cond base --task syntax --n 30
  python src/run_vlm.py --cond rag  --task stenosis --n 30
Works on CPU (slow) or GPU (Colab T4: add --device cuda).
"""
import argparse, json, random, re, time
from pathlib import Path

import torch
from PIL import Image
from transformers import AutoModelForImageTextToText, AutoProcessor

from gt import build_syntax_gt, build_stenosis_gt
from rag import Retriever

SYSTEM = "You are an experienced interventional cardiologist reading a single coronary angiography frame."

SCHEMA = """Describe this coronary angiogram as JSON with exactly these keys:
"side": "left" or "right" or "unknown" - which coronary system is injected,
"arteries": a list of {{"artery": one of "LM", "LAD", "LCx", "RCA", "point": [x, y]}} for every artery that is clearly visible; point is a pixel on that artery,
"stenoses": a list of {{"artery": one of "LM", "LAD", "LCx", "RCA", "point": [x, y], "severity": "mild" or "moderate" or "severe" or "occlusion", "confidence": a number from 0 to 1}}; point is the center of the narrowing,
"not_assessable": a list of short notes about what cannot be judged from this image.
The image is {w}x{h} pixels; coordinates are pixels in this image. Use only what is visible; if unsure, leave the list empty or say so in "not_assessable". Output only the JSON."""

RAG_QUERY = ("left right coronary injection laterality LM LAD LCx RCA segments course "
             "stenosis narrowing location projection report not visible")


def parse_json(text):
    text = re.sub(r"^```(json)?|```$", "", text.strip(), flags=re.M).strip()
    m = re.search(r"\{.*\}", text, flags=re.S)
    if not m:
        return None
    try:
        return json.loads(m.group(0))
    except json.JSONDecodeError:
        return None


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--model", default="Qwen/Qwen2.5-VL-3B-Instruct")
    ap.add_argument("--cond", choices=["base", "rag"], default="base")
    ap.add_argument("--task", choices=["syntax", "stenosis"], default="syntax")
    ap.add_argument("--split", default="test")
    ap.add_argument("--n", type=int, default=30)
    ap.add_argument("--k", type=int, default=6, help="retrieved chunks for RAG")
    ap.add_argument("--seed", type=int, default=0)
    ap.add_argument("--root", default="/home/user/data/arcade")
    ap.add_argument("--device", default="cuda" if torch.cuda.is_available() else "cpu")
    ap.add_argument("--out", default="results")
    ap.add_argument("--dtype", choices=["bf16", "fp16", "fp32"], default="bf16", help="T4 GPU: fp16")
    ap.add_argument("--load-4bit", action="store_true")
    args = ap.parse_args()

    gts = build_syntax_gt(args.root, args.split) if args.task == "syntax" else build_stenosis_gt(args.root, args.split)
    random.Random(args.seed).shuffle(gts)
    gts = gts[: args.n]

    context = ""
    if args.cond == "rag":
        chunks = Retriever(str(Path(__file__).resolve().parent.parent / "kb" / "knowledge.jsonl")).search(RAG_QUERY, k=args.k)
        context = "Reference knowledge:\n" + "\n".join(f"- {c['text']}" for c in chunks) + "\n\n"

    dtype = {"bf16": torch.bfloat16, "fp16": torch.float16, "fp32": torch.float32}[args.dtype]
    if args.load_4bit:  # e.g. 7B model on a free Colab T4
        from transformers import BitsAndBytesConfig
        q = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_compute_dtype=torch.float16, bnb_4bit_quant_type="nf4")
        model = AutoModelForImageTextToText.from_pretrained(args.model, quantization_config=q, device_map="auto").eval()
    else:
        model = AutoModelForImageTextToText.from_pretrained(args.model, dtype=dtype).to(args.device).eval()
    processor = AutoProcessor.from_pretrained(args.model, min_pixels=256 * 28 * 28, max_pixels=512 * 512)

    tag = f"{Path(args.model).name}_{args.cond}_{args.task}_{args.split}_n{args.n}"
    out_path = Path(args.out) / f"pred_{tag}.jsonl"
    done = set()
    if out_path.exists():
        done = {json.loads(l)["image_id"] for l in open(out_path)}
    fout = open(out_path, "a")

    for i, g in enumerate(gts):
        if g["image_id"] in done:
            continue
        img = Image.open(g["file"]).convert("RGB")
        # size the model actually sees (Qwen resizes to multiples of 28)
        probe = processor.image_processor(images=[img], return_tensors="pt")
        gh, gw = probe["image_grid_thw"][0][1:].tolist()
        rh, rw = gh * 14, gw * 14
        prompt = context + SCHEMA.format(w=rw, h=rh)
        messages = [
            {"role": "system", "content": [{"type": "text", "text": SYSTEM}]},
            {"role": "user", "content": [{"type": "image"}, {"type": "text", "text": prompt}]},
        ]
        text = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
        inputs = processor(text=[text], images=[img], return_tensors="pt").to(args.device)
        t0 = time.time()
        with torch.no_grad():
            out = model.generate(**inputs, max_new_tokens=350, do_sample=False)
        raw = processor.batch_decode(out[:, inputs["input_ids"].shape[1]:], skip_special_tokens=True)[0]
        rec = {"image_id": g["image_id"], "raw": raw, "parsed": parse_json(raw),
               "scale": [g["width"] / rw, g["height"] / rh], "sec": round(time.time() - t0, 1)}
        fout.write(json.dumps(rec, ensure_ascii=False) + "\n")
        fout.flush()
        print(f"[{i+1}/{len(gts)}] id={g['image_id']} {rec['sec']}s parsed={rec['parsed'] is not None}", flush=True)


if __name__ == "__main__":
    main()


In [ ]:
%%writefile /content/angio-vlm/src/evaluate.py
"""Evaluate VLM predictions against ARCADE ground truth.

Syntax subset  -> laterality, artery-level P/R/F1, grounding of artery claims on segment masks.
Stenosis subset -> pointing precision/recall of stenosis claims on stenosis boxes.
Each metric is compared with a trivial baseline (majority / random point).
"""
import argparse, json, random
from collections import Counter
from pathlib import Path

import numpy as np
from scipy.ndimage import binary_dilation

from gt import ARTERIES, build_stenosis_gt, build_syntax_gt, syntax_artery_masks

LEFT = {"LM", "LAD", "LCx"}
TOL_PX = 7      # tolerance around a vessel mask, pixels (512x512 image)
BOX_PAD = 10    # tolerance around a stenosis box, pixels


def load_preds(path):
    return {r["image_id"]: r for r in map(json.loads, open(path))}


def norm_artery(a):
    a = str(a).strip().upper()
    return {"LCX": "LCx", "CX": "LCx", "CIRC": "LCx", "LMCA": "LM", "LEFT MAIN": "LM"}.get(a, a if a in {"LM", "LAD", "RCA"} else None)


def point_of(item, scale):
    try:
        x, y = item["point"][:2]
        return float(x) * scale[0], float(y) * scale[1]
    except Exception:
        return None


def on_mask(mask, p):
    if p is None:
        return False
    x, y = int(round(p[0])), int(round(p[1]))
    return 0 <= y < mask.shape[0] and 0 <= x < mask.shape[1] and bool(mask[y, x])


def prf(tp, fp, fn):
    p = tp / (tp + fp) if tp + fp else 0.0
    r = tp / (tp + fn) if tp + fn else 0.0
    f = 2 * p * r / (p + r) if p + r else 0.0
    return round(p, 3), round(r, 3), round(f, 3)


def eval_syntax(pred_path, root, split):
    preds = load_preds(pred_path)
    gts = {g["image_id"]: g for g in build_syntax_gt(root, split)}
    ids = [i for i in preds if i in gts]
    st = Counter()
    vessel_area = []
    examples = []
    for iid in ids:
        g, r = gts[iid], preds[iid]
        p = r["parsed"]
        st["n"] += 1
        if p is None:
            st["unparsed"] += 1
            p = {}
        side = str(p.get("side", "unknown")).lower()
        st["side_unknown"] += side not in {"left", "right"}
        st["side_correct"] += side == g["side"]
        masks = {a: binary_dilation(m, iterations=TOL_PX) for a, m in syntax_artery_masks(root, split, iid).items()}
        any_vessel = np.any(np.stack(list(masks.values())), axis=0)
        vessel_area.append(any_vessel.mean())
        pred_arts = {}
        for it in p.get("arteries", []) or []:
            a = norm_artery(it.get("artery", "")) if isinstance(it, dict) else None
            if a and a not in pred_arts:
                pred_arts[a] = point_of(it, r["scale"])
        gset = set(g["arteries"])
        for a in ARTERIES:
            st["tp"] += a in pred_arts and a in gset
            st["fp"] += a in pred_arts and a not in gset
            st["fn"] += a not in pred_arts and a in gset
        # claims about the other coronary system (left vs right) - anatomical contradiction
        gt_left = g["side"] == "left"
        st["claims"] += len(pred_arts)
        st["cross_system"] += sum((a in LEFT) != gt_left for a in pred_arts)
        for a, pt in pred_arts.items():
            st["pts_on_vessel"] += on_mask(any_vessel, pt)
            if a in gset:
                st["claims_true"] += 1
                st["pts_on_own_artery"] += on_mask(masks[a], pt)
        supported = sum(a in gset and on_mask(masks[a], pt) for a, pt in pred_arts.items())
        st["supported"] += supported
        examples.append({"image_id": iid, "gt": g["arteries"], "pred": {a: pt for a, pt in pred_arts.items()}, "side": side})
    n = st["n"]
    P, R, F = prf(st["tp"], st["fp"], st["fn"])
    # baselines on the same images
    maj_side = Counter(gts[i]["side"] for i in ids).most_common(1)[0]
    rng = random.Random(0)
    bp = Counter()
    for iid in ids:  # baseline: always predict LM + LAD (most frequent arteries)
        gset = set(gts[iid]["arteries"])
        for a in ARTERIES:
            pa = a in {"LM", "LAD"}
            bp["tp"] += pa and a in gset; bp["fp"] += pa and a not in gset; bp["fn"] += (not pa) and a in gset
    bP, bR, bF = prf(bp["tp"], bp["fp"], bp["fn"])
    return {
        "images": n,
        "parse_rate": round(1 - st["unparsed"] / n, 3),
        "side_accuracy": round(st["side_correct"] / n, 3),
        "side_unknown_rate": round(st["side_unknown"] / n, 3),
        "side_majority_baseline": round(maj_side[1] / n, 3),
        "artery_P_R_F1": [P, R, F],
        "artery_baseline_LM_LAD_P_R_F1": [bP, bR, bF],
        "artery_claims_per_image": round(st["claims"] / n, 2),
        "cross_system_claim_rate": round(st["cross_system"] / max(st["claims"], 1), 3),
        "point_on_any_vessel_rate": round(st["pts_on_vessel"] / max(st["claims"], 1), 3),
        "random_point_on_vessel_baseline": round(float(np.mean(vessel_area)), 3),
        "point_on_correct_artery_rate": round(st["pts_on_own_artery"] / max(st["claims_true"], 1), 3),
        "supported_claim_rate": round(st["supported"] / max(st["claims"], 1), 3),
    }, examples


def eval_stenosis(pred_path, root, split):
    preds = load_preds(pred_path)
    gts = {g["image_id"]: g for g in build_stenosis_gt(root, split)}
    ids = [i for i in preds if i in gts]
    st = Counter()
    area = []
    confs_hit, confs_miss = [], []
    sev = Counter()
    examples = []
    for iid in ids:
        g, r = gts[iid], preds[iid]
        p = r["parsed"] or {}
        st["n"] += 1
        st["unparsed"] += r["parsed"] is None
        boxes = [(x - BOX_PAD, y - BOX_PAD, x + w + BOX_PAD, y + h + BOX_PAD) for x, y, w, h in g["stenosis_boxes"]]
        m = np.zeros((g["height"], g["width"]), bool)
        for x0, y0, x1, y1 in boxes:
            m[max(0, int(y0)):int(y1), max(0, int(x0)):int(x1)] = True
        area.append(m.mean())
        art_pts = {tuple(point_of(it, r["scale"]) or ()) for it in (p.get("arteries") or []) if isinstance(it, dict)}
        sts = [it for it in (p.get("stenoses") or []) if isinstance(it, dict)]
        st["gt"] += len(boxes)
        st["claims"] += len(sts)
        st["no_claim_images"] += len(sts) == 0
        hit_boxes = set()
        pts = []
        for it in sts:
            pt = point_of(it, r["scale"])
            pts.append(pt)
            sev[str(it.get("severity", "?"))] += 1
            st["copied_artery_point"] += pt is not None and tuple(pt) in art_pts
            hits = [j for j, (x0, y0, x1, y1) in enumerate(boxes) if pt and x0 <= pt[0] <= x1 and y0 <= pt[1] <= y1]
            st["hit_claims"] += bool(hits)
            hit_boxes.update(hits)
            try:
                (confs_hit if hits else confs_miss).append(float(it.get("confidence")))
            except (TypeError, ValueError):
                pass
        st["gt_found"] += len(hit_boxes)
        examples.append({"image_id": iid, "pred_points": pts, "gt_boxes": g["stenosis_boxes"]})
    n = st["n"]
    return {
        "images": n,
        "parse_rate": round(1 - st["unparsed"] / n, 3),
        "gt_stenoses": st["gt"],
        "claims_per_image": round(st["claims"] / n, 2),
        "images_without_claims": round(st["no_claim_images"] / n, 3),
        "point_precision": round(st["hit_claims"] / max(st["claims"], 1), 3),
        "random_point_precision_baseline": round(float(np.mean(area)), 3),
        "stenosis_recall": round(st["gt_found"] / max(st["gt"], 1), 3),
        "stenosis_point_copied_from_artery": round(st["copied_artery_point"] / max(st["claims"], 1), 3),
        "severity_counts": dict(sev),
        "mean_conf_hit": round(float(np.mean(confs_hit)), 3) if confs_hit else None,
        "mean_conf_miss": round(float(np.mean(confs_miss)), 3) if confs_miss else None,
    }, examples


if __name__ == "__main__":
    ap = argparse.ArgumentParser()
    ap.add_argument("--root", default="/home/user/data/arcade")
    ap.add_argument("--split", default="test")
    ap.add_argument("--results", default="results")
    args = ap.parse_args()
    out = {}
    for f in sorted(Path(args.results).glob("pred_*.jsonl")):
        task = "syntax" if "_syntax_" in f.name else "stenosis"
        fn = eval_syntax if task == "syntax" else eval_stenosis
        metrics, ex = fn(f, args.root, args.split)
        out[f.stem.replace("pred_", "")] = metrics
        json.dump(ex, open(Path(args.results) / f"examples_{f.stem[5:]}.json", "w"))
    json.dump(out, open(Path(args.results) / "metrics.json", "w"), indent=1, ensure_ascii=False)
    print(json.dumps(out, indent=1, ensure_ascii=False))


In [ ]:
%%writefile /content/angio-vlm/src/viz.py
"""Overlay ground truth (ARCADE masks / boxes) and VLM claims (points) for a few images."""
import argparse, json
from pathlib import Path

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
from PIL import Image

from gt import build_stenosis_gt, build_syntax_gt, syntax_artery_masks

COLORS = {"LM": "#e6194b", "LAD": "#3cb44b", "LCx": "#4363d8", "RCA": "#f58231"}


def show_syntax(ax, root, split, g, ex):
    img = np.array(Image.open(g["file"]).convert("L"))
    ax.imshow(img, cmap="gray")
    for a, m in syntax_artery_masks(root, split, g["image_id"]).items():
        rgba = np.zeros((*m.shape, 4))
        rgba[m] = matplotlib.colors.to_rgba(COLORS[a], 0.35)
        ax.imshow(rgba)
    for a, pt in ex["pred"].items():
        if pt:
            ax.scatter([pt[0]], [pt[1]], s=90, marker="X", c=COLORS.get(a, "w"), edgecolors="white", linewidths=1.2)
            ax.annotate(a, pt, xytext=(6, -6), textcoords="offset points", color="white", fontsize=8, weight="bold")
    ax.set_title(f"#{g['image_id']}  GT: {', '.join(g['arteries'])}\nVLM: {', '.join(ex['pred']) or '-'} | side={ex['side']}", fontsize=8)
    ax.axis("off")


def show_stenosis(ax, g, ex):
    img = np.array(Image.open(g["file"]).convert("L"))
    ax.imshow(img, cmap="gray")
    for x, y, w, h in g["stenosis_boxes"]:
        ax.add_patch(plt.Rectangle((x, y), w, h, fill=False, ec="#3cb44b", lw=1.8))
    for pt in ex["pred_points"]:
        if pt:
            ax.scatter([pt[0]], [pt[1]], s=90, marker="X", c="#e6194b", edgecolors="white", linewidths=1.2)
    ax.set_title(f"#{g['image_id']}  GT stenoses: {len(g['stenosis_boxes'])} | VLM points: {len(ex['pred_points'])}", fontsize=8)
    ax.axis("off")


if __name__ == "__main__":
    ap = argparse.ArgumentParser()
    ap.add_argument("--root", default="/home/user/data/arcade")
    ap.add_argument("--split", default="test")
    ap.add_argument("--tag", required=True, help="e.g. Qwen2.5-VL-3B-Instruct_rag_syntax_test_n50")
    ap.add_argument("--k", type=int, default=6)
    ap.add_argument("--out", default="results/figures")
    args = ap.parse_args()
    ex = json.load(open(f"results/examples_{args.tag}.json"))[: args.k]
    syntax = "_syntax_" in args.tag
    gts = {g["image_id"]: g for g in (build_syntax_gt if syntax else build_stenosis_gt)(args.root, args.split)}
    cols = 3
    rows = (len(ex) + cols - 1) // cols
    fig, axes = plt.subplots(rows, cols, figsize=(cols * 3.4, rows * 3.7))
    for ax, e in zip(np.ravel(axes), ex):
        if syntax:
            show_syntax(ax, args.root, args.split, gts[e["image_id"]], e)
        else:
            show_stenosis(ax, gts[e["image_id"]], e)
    for ax in np.ravel(axes)[len(ex):]:
        ax.axis("off")
    if syntax:
        handles = [plt.Line2D([], [], marker="s", ls="", color=c, label=a) for a, c in COLORS.items()]
        fig.legend(handles=handles, loc="lower center", ncol=4, fontsize=8,
                   title="маска = разметка ARCADE, X = точка, указанная VLM", title_fontsize=8)
    else:
        fig.legend(handles=[plt.Line2D([], [], color="#3cb44b", label="стеноз по разметке ARCADE"),
                            plt.Line2D([], [], marker="X", ls="", color="#e6194b", label="точка стеноза от VLM")],
                   loc="lower center", ncol=2, fontsize=8)
    fig.tight_layout(rect=(0, 0.06, 1, 1))
    Path(args.out).mkdir(parents=True, exist_ok=True)
    fig.savefig(Path(args.out) / f"{args.tag}.png", dpi=130)
    print("saved", Path(args.out) / f"{args.tag}.png")


In [ ]:
%%writefile /content/angio-vlm/kb/knowledge.jsonl
{"id": "side-1", "topic": "laterality", "text": "Left coronary angiogram (LCA injection): the catheter tip engages the left main ostium; contrast fills the left main (LM), which bifurcates into the LAD and the LCx. Two or more large branches diverge from a short common trunk."}
{"id": "side-2", "topic": "laterality", "text": "Right coronary angiogram (RCA injection): a single long vessel with a C-shaped (LAO view) or L-shaped (RAO view) course along the atrioventricular groove; no bifurcation into two large branches near the catheter tip. Distally it gives the posterior descending artery (PDA) and posterolateral branches."}
{"id": "side-3", "topic": "laterality", "text": "In a single injection only one coronary system is opacified: either the left system (LM, LAD, LCx) or the right system (RCA). Do not report LAD or LCx in a right coronary injection, and do not report RCA in a left coronary injection."}
{"id": "seg-rca", "topic": "anatomy", "text": "SYNTAX segments of the RCA: 1 proximal RCA (ostium to half of the way to the acute margin), 2 mid RCA, 3 distal RCA (to the origin of the PDA), 4 posterior descending artery (PDA), 16, 16a, 16b, 16c posterolateral branches from the RCA."}
{"id": "seg-lm", "topic": "anatomy", "text": "SYNTAX segment 5: left main coronary artery (LM), from the ostium to the bifurcation into LAD and LCx. It is short (about 1-2 cm) and lies closest to the catheter tip in a left injection."}
{"id": "seg-lad", "topic": "anatomy", "text": "SYNTAX segments of the LAD: 6 proximal LAD (to the first major septal), 7 mid LAD (to the second diagonal), 8 apical LAD; 9 and 9a first diagonal branches, 10 and 10a second diagonal branches. The LAD runs in the anterior interventricular groove toward the apex and gives septal branches (downward) and diagonal branches (laterally)."}
{"id": "seg-lcx", "topic": "anatomy", "text": "SYNTAX segments of the LCx: 11 proximal circumflex, 13 distal circumflex, 12, 12a, 12b obtuse marginal branches, 14, 14a, 14b left posterolateral branches, 15 posterior descending branch from the LCx (in left dominance). The LCx runs in the left atrioventricular groove."}
{"id": "proj-1", "topic": "projection", "text": "Standard projections: RAO caudal shows the LM bifurcation, proximal LCx and obtuse marginals well; RAO cranial and AP cranial show the mid and distal LAD with diagonals separated; LAO caudal (spider view) shows the LM and proximal LAD/LCx; LAO for the RCA shows a C-shaped proximal and mid RCA."}
{"id": "proj-2", "topic": "projection", "text": "Orientation cues: the LAD usually courses toward the apex along the anterior interventricular groove and is the vessel with septal branches; the LCx curves along the atrioventricular groove and gives marginal branches toward the lateral wall. Overlapping vessels and foreshortening make segment boundaries uncertain."}
{"id": "dom-1", "topic": "dominance", "text": "Coronary dominance is defined by the artery that gives the PDA: right dominance (about 70-80% of people, PDA from RCA, segment 4), left dominance (PDA from LCx, segment 15), co-dominance. Dominance cannot be determined reliably from a single injection."}
{"id": "sten-1", "topic": "stenosis", "text": "Angiographic stenosis appears as a focal narrowing of the contrast-filled lumen relative to adjacent reference segments: an hourglass narrowing, an abrupt step-down in caliber, a filling defect, or an abrupt vessel cut-off (occlusion). Gradual tapering toward the distal vessel is normal and is not a stenosis."}
{"id": "sten-2", "topic": "stenosis", "text": "Stenosis severity by visual diameter estimation: below 50% mild (not significant), 50-69% moderate, 70-99% severe, 100% total occlusion. A stenosis of 50% or more in the left main, or 70% or more in other epicardial vessels, is considered significant."}
{"id": "sten-3", "topic": "stenosis", "text": "Pitfalls that mimic stenosis: vessel overlap, foreshortening, branch origins, catheter-induced spasm, myocardial bridging (systolic compression of the LAD), and streaming of incompletely mixed contrast. Report a stenosis only when the narrowing is visible on the image."}
{"id": "report-1", "topic": "reporting", "text": "Structured angiographic report: for each artery and segment state whether it is visualized, whether a stenosis is present, its estimated severity, and its location. If a segment is not visible in the projection, state that it cannot be assessed instead of guessing."}
{"id": "report-2", "topic": "reporting", "text": "Each finding should be tied to a specific location on the image. When the image quality, contrast filling or projection is insufficient, the finding should be marked as uncertain or not assessable."}


In [ ]:
%cd /content/angio-vlm
!mkdir -p /content/data && cd /content/data && wget -q -O arcade.zip https://zenodo.org/api/records/10390295/files/arcade.zip/content && unzip -q -o arcade.zip && ls arcade

## Прогон
`N = 300` — вся тестовая часть. Если сессия оборвётся, просто запусти ячейку снова: уже готовые снимки пропускаются (но файлы живут только пока жива сессия).

In [ ]:
ROOT = "/content/data/arcade"
N = 300
RUNS = [("Qwen/Qwen2.5-VL-3B-Instruct", "--dtype fp16"),
        ("Qwen/Qwen2.5-VL-7B-Instruct", "--load-4bit")]
for model, extra in RUNS:
    for cond in ["base", "rag"]:
        for task in ["syntax", "stenosis"]:
            !python src/run_vlm.py --model $model --cond $cond --task $task --n $N --root $ROOT $extra --device cuda 2>&1 | grep -E "^\[|Error" | tail -3

In [ ]:
!python src/evaluate.py --root $ROOT

In [ ]:
import glob, os
from IPython.display import Image, display
for f in glob.glob('results/examples_*.json'):
    tag = os.path.basename(f)[9:-5]
    !python src/viz.py --root $ROOT --tag $tag
for f in sorted(glob.glob('results/figures/*.png')):
    print(f); display(Image(f))

In [ ]:
!zip -qr /content/results.zip results
from google.colab import files
files.download('/content/results.zip')